# Make climate change figure for poster

## Imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time
import cartopy.crs as ccrs

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Load data

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## Compute

### mean state / climatology

In [ ]:
t0 = dict(time=slice("1851","1880"))
t1 = dict(time=slice("2061","2090"))

### spatial patterns

In [ ]:
## average
clim0 = forced.sel(t0).mean("time")
clim1 = forced.sel(t1).mean("time")
dclim = clim1-clim0

## reconstruct
var_list = ["sst", "T", "w"]
var_list = var_list + [f"{v}_comp" for v in var_list]
clim0_spatial = src.utils.reconstruct_wrapper(clim0[var_list])
clim1_spatial = src.utils.reconstruct_wrapper(clim1[var_list])

### tropical SST

In [ ]:
## tropical sst for model
trop_sst = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/trop_sst.nc"))
trop_sst = trop_sst["trop_sst_15"].sel(time=forced.time)
trop_sst0 = trop_sst.sel(t0).mean()
trop_sst1 = trop_sst.sel(t1).mean()

### relative SST

In [ ]:
## get relative sst
clim0_spatial["rsst"] = clim0_spatial["sst"] - trop_sst0
clim1_spatial["rsst"] = clim1_spatial["sst"] - trop_sst1

### Get difference

In [ ]:
dclim_spatial = clim1_spatial-clim0_spatial

### ENSO pattern

In [ ]:
def get_djf_enso(data):
    """Get DJF pattern for ENSO"""
    ## resample to djf
    data_seasonal = data["sst"].resample({"time":"QS-DEC"}).mean()

    ## get djf
    is_djf = data_seasonal.time.dt.month == 12
    data_djf = data_seasonal.sel(time=is_djf).isel(time=slice(1,-1))

    ## get Niño 3.4 index
    T34 = src.utils.reconstruct_fn(
        scores=data_djf,
        components=data["sst_comp"],
        fn=src.utils.get_nino34,
    )

    ## standardize it
    T34 = T34 / T34.std()

    ## merge data
    XY = xr.merge([T34.rename("T34"), data_djf])

    ## get coefficients
    m_proj = src.utils.regress_xr_proj(XY, x_vars=["T34"], y_vars=["sst"])
    m_proj = m_proj["sst"].squeeze(drop=True)

    ## reconstruct
    m = src.utils.reconstruct_fn(
        scores=m_proj,
        components=data["sst_comp"],
        fn =lambda x : x,
    )
        

    return m

In [ ]:
#### Get ENSO pattern
enso_pattern0 = get_djf_enso(anom.sel(t0))

## Plot SST

In [ ]:
## specify lons/lats for E/W boxes
LONS_E = [240, 280]
LONS_W = [150, 190]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-20, 0, 20],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[150, -170, -120, -80],
            ylocs=[-20, 0, 20],
        )
        gl_.top_labels = False
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return

import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_sst2(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

def plot_sst_sigma(ax, data, lev_min=.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min+dx*(nlev+1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

In [ ]:
3.5*7/6

In [ ]:
## specify intervals
dxs = np.array([.8, 0.4])

fig = plt.figure(figsize=(6, 5), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=21, lon_range=(140, 285))
axs = src.utils.subplots_with_proj(fig, nrows=2, ncols=1, format_func=format_func)

cp00 = plot_sst2(axs[0, 0], clim0_spatial["rsst"], dx=dxs[0])
cp01 = plot_sst2(axs[1, 0], enso_pattern0, dx=dxs[1])
# cp01 = plot_sst_sigma(axs[1, 0], sst_sigma0, dx=dxs[1])

## add formatting to axs
for ax in axs.flatten():
    plot_wbox(ax, c="gray", lw=0.8)
    plot_ebox(ax, c="gray", lw=0.8)
    src.utils.plot_nino34_box(ax, c="w", ls="--", lw=0.8, alpha=0.75)

## label
axs[0, 0].set_title("(a) Relative SST mean", loc="left")
axs[1, 0].set_title(
    r"(b) SST ENSO pattern", loc="left"
)

## add labels
add_gridlines(axs)

bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax, dx in zip(axs.flatten(), dxs.flatten()):
    ax.text(s=f"Interval: {dx} K", transform=ax.transAxes, **legend_kwargs)
    ax.set_aspect("auto")

## SAVE
# save(fig, "spatial-valid")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.5), layout="constrained")

## temperature
cp_T = ax.contourf(
    clim0_spatial["T"].longitude,
    clim0_spatial.z_t,
    clim0_spatial["T"],
    cmap="cmo.thermal",
    # levels=np.arange(12,30,2),
    levels=np.arange(10,31,3),
    extend="both",
)

# ## upwelling
cp_T = ax.contour(
    clim0_spatial["w"].longitude,
    clim0_spatial.z_t,
    clim0_spatial["w"],
    colors="k",
    alpha=.9,
    levels = src.utils.make_cb_range(100, 10),
    extend="both",
    linewidths=1,
)

## plot 80-m MLD
ax.add_patch(
    mpl.patches.Rectangle(
        height=-80, linewidth=2,  ls="-", xy=(210,80), width=60, edgecolor="w", fill=None,
    ),
)

ax.axhline(50, ls="--", c="w", lw=1.5)



## label / formatting
src.utils.label_subplot(ax=ax, posn=(0.03, 0.05), label="(c)")
ax.set_xlim([140, 285])
ax.set_ylim([250,5])
ax.set_xticks([140, 210, 280], labels=[r"140$^{\circ}$E",  r"150$^{\circ}$W", r"80$^{\circ}$W"])
ax.set_yticks([5,50, 150, 250], labels=[0, 50, 150, 250])

# ## colorbars
# fig.colorbar(cp_T, ticks=[-5, 0, 5], label=r"$^{\circ}$C", orientation="horizontal")
# fig.colorbar(cp_w, ticks=[-20, 0, 20], label=r"m / month", orientation="horizontal")

plt.show()